# 05. 무대 위에서 — **통합**

## 이 노트북의 위치

```
기반     : NB01 (듣다)
확장형   : NB02 (시각) + NB03 (음악)
협업형   : NB04 (AI와 대화)
통합 ✦   : NB05 (무대) ← 여기
```

확장과 협업의 모든 산출물을 **하나의 미니 공연으로 엮습니다**.

## 재료
- 원 연주 2곡 (Satie, Prokofiev)
- NB02 학생의 cue 매핑 (확장: 시각)
- NB03 변주 MIDI (확장: 음악)
- NB04 해석 피드백 + 매핑 제안 (협업)

## 학생이 하는 것
**타임라인을 직접 편집해서** 공연의 흐름을 설계합니다.
어느 곡을 언제, 어떤 매핑으로, AI 응답을 어디서 띄울지.

이 편집 자체가 **학생과 AI의 마지막 협업**입니다.

In [ ]:
from pathlib import Path
import json

ARTIFACTS = Path('artifacts')
ASSETS = Path('assets')

# 필수 산출물 확인
required = {
    'satie_audio': ASSETS / 'satie_gymnopedie_no1.wav',
    'prokofiev_audio': ASSETS / 'prokofiev_toccata_op11.wav',
    'satie_cues': ARTIFACTS / 'cues' / 'satie_visual_cues.json',
    'prokofiev_cues': ARTIFACTS / 'cues' / 'prokofiev_visual_cues.json',
}
optional = {
    'mapping_suggestions': ARTIFACTS / 'responses' / 'mapping_suggestions.json',
    'compare_interpretive': ARTIFACTS / 'responses' / 'compare_interpretive.md',
}

print('필수 산출물:')
for k, p in required.items():
    ok = '✅' if p.exists() else '❌'
    print(f'  {ok} {k}: {p}')

print('\n선택 산출물:')
for k, p in optional.items():
    ok = '✅' if p.exists() else '⚠️ 없음'
    print(f'  {ok} {k}: {p}')

missing = [k for k, p in required.items() if not p.exists()]
if missing:
    raise FileNotFoundError(f'다음이 없습니다: {missing}. NB01-04를 먼저 실행하세요.')

---
## 1. 공연 타임라인 편집

아래 `timeline` 리스트를 직접 편집하세요. 각 섹션은 **source + mapping + optional text overlay**입니다.

### source 종류
- `satie`, `prokofiev`: 원 연주
- NB03 변주 MIDI (파일명으로 직접 참조)

### mapping 종류
- `서정적`, `역동적`, `미니멀`: NB02 프리셋
- `claude_satie`, `claude_prokofiev`: NB04 Mode C의 AI 제안 매핑
- `custom`: 직접 정의

In [ ]:
# ★ 여기를 편집하세요 ★
timeline = [
    {
        'section': '도입',
        'source': 'satie',
        'start': 0.0,
        'duration': 45.0,
        'mapping': '서정적',
        'text_overlay': None,
    },
    {
        'section': '전환',
        'source': 'satie',
        'start': 45.0,
        'duration': 15.0,
        'mapping': 'claude_satie',  # Claude 제안 적용
        'text_overlay': 'AI의 시각 해석',
    },
    {
        'section': '폭발',
        'source': 'prokofiev',
        'start': 0.0,
        'duration': 60.0,
        'mapping': '역동적',
        'text_overlay': None,
    },
    {
        'section': '대비',
        'source': 'prokofiev',
        'start': 60.0,
        'duration': 20.0,
        'mapping': 'claude_prokofiev',
        'text_overlay': 'AI의 시각 해석',
    },
]

total = sum(s['duration'] for s in timeline)
print(f'⏱️  총 재생 시간: {total:.1f}초 ({total/60:.1f}분)')
print()
print('섹션 구성:')
cursor = 0.0
for s in timeline:
    print(f"  {cursor:5.1f}s → {cursor + s['duration']:5.1f}s | {s['section']:6s} | {s['source']:10s} | mapping={s['mapping']:20s}" + (f" | {s['text_overlay']}" if s['text_overlay'] else ''))
    cursor += s['duration']

---
## 2. 타임라인 시각화

각 섹션의 cue 곡선을 이어 붙여 공연 전체의 흐름을 미리 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cues_cache = {
    'satie': json.loads(required['satie_cues'].read_text()),
    'prokofiev': json.loads(required['prokofiev_cues'].read_text()),
}

fps = cues_cache['satie']['fps']
channels = ['energy', 'density', 'harmonic_tension', 'register_spread']
colors = {'satie': '#4f46e5', 'prokofiev': '#dc2626'}

fig, axes = plt.subplots(len(channels), 1, figsize=(14, 8), sharex=True)
global_t = 0.0
for section in timeline:
    src = section['source']
    if src not in cues_cache:
        continue
    frames = cues_cache[src]['frames']
    start_frame = int(section['start'] * fps)
    end_frame = int((section['start'] + section['duration']) * fps)
    segment = frames[start_frame:end_frame]
    seg_t = np.arange(len(segment)) / fps + global_t
    for ax, ch in zip(axes, channels):
        vals = [f[ch] for f in segment]
        ax.fill_between(seg_t, vals, alpha=0.6, color=colors.get(src, '#888'))
    # 섹션 경계
    for ax in axes:
        ax.axvline(global_t, color='black', linestyle='--', alpha=0.3, linewidth=0.8)
    axes[0].text(global_t + 0.3, 0.95, section['section'], fontsize=10, fontweight='bold', transform=axes[0].get_xaxis_transform())
    global_t += section['duration']

for ax, ch in zip(axes, channels):
    ax.set_ylabel(ch)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('공연 시간 (초)')
axes[0].set_title('공연 타임라인 — cue 흐름')
plt.tight_layout()
plt.show()

---
## 3. 공연 JSON 내보내기

웹 앱(`pianokit_web/`)의 `/stage` 라우트에서 이 JSON을 소비합니다.

In [ ]:
# 매핑 정보 병합
mapping_suggestions = {}
if optional['mapping_suggestions'].exists():
    mapping_suggestions = json.loads(optional['mapping_suggestions'].read_text())

performance = {
    'title': 'pianokit performance — Satie × Prokofiev',
    'total_duration': total,
    'timeline': timeline,
    'sources': {
        'satie': {'audio': str(required['satie_audio']), 'cues': str(required['satie_cues'])},
        'prokofiev': {'audio': str(required['prokofiev_audio']), 'cues': str(required['prokofiev_cues'])},
    },
    'claude_mapping_suggestions': mapping_suggestions,
}

out_path = ARTIFACTS / 'performance_timeline.json'
out_path.write_text(json.dumps(performance, ensure_ascii=False, indent=2))
print(f'✅ 공연 JSON 저장: {out_path}')
print(f'   {len(timeline)}개 섹션, 총 {total:.1f}초')

---
## 4. 웹 앱 연동

`pianokit_web/public/performance_timeline.json`으로 복사하면 웹 앱이 소비할 수 있습니다.

In [ ]:
import shutil

web_public = Path('pianokit_web') / 'public'
if web_public.exists():
    # 타임라인 + 오디오 + cues 복사
    web_data = web_public / 'stage_data'
    web_data.mkdir(exist_ok=True)
    shutil.copy(out_path, web_data / 'performance_timeline.json')
    for key in ['satie', 'prokofiev']:
        shutil.copy(required[f'{key}_audio'], web_data / f'{key}.wav')
        shutil.copy(required[f'{key}_cues'], web_data / f'{key}_cues.json')
    print(f'✅ 웹 앱으로 복사됨: {web_data}')
    print(f'   웹 앱에서 fetch("/stage_data/performance_timeline.json")로 접근 가능')
else:
    print(f'⚠️ {web_public} 없음. 웹 앱 submodule이 체크아웃되어 있는지 확인하세요.')
    print(f'   수동 복사: cp {out_path} pianokit_web/public/stage_data/')

---
## 공연 체크리스트

조별 발표 전 확인:

1. ✅ 타임라인의 각 섹션이 음악적 흐름을 가지는가?
2. ✅ 각 구간의 매핑 선택이 음악 성격과 맞는가? (확장의 품질)
3. ✅ AI 제안 매핑을 **수용/거부**한 구간이 있는가? (협업의 흔적)
4. ✅ 두 곡의 대비가 공연 전체의 드라마를 만드는가?

## 확장 과제

- NB03 변주 MIDI를 타임라인에 삽입하기
- 같은 구간에서 **학생 매핑 vs AI 매핑**을 비교하는 A/B 섹션 만들기
- 조끼리 타임라인을 교환해서 다시 편집 (확장과 협업의 재조합)

---

## 워크숍 마무리 질문

- **확장**: 내 연주의 어떤 차원이 가장 설득력 있게 확장되었는가?
- **협업**: AI 제안 중 수용한 것과 거부한 것은 무엇이고, 왜?